In [1]:
"""Load the raw data.

Path is relative to notebook/, so it survives the repo being moved or cloned — the old
absolute path also pointed at data/Country-data.csv, which no longer exists after the
data folder restructure.

`df` stays the untouched raw frame from here on; every transform builds a new name.
"""
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path('..') / 'data' / 'raw_data' / 'Country-data.csv'
df = pd.read_csv(RAW)
df.head()

,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200


In [2]:
df


,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
0,Afghanistan,90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
1,Albania,16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
2,Algeria,27.3,38.4,4.17,31.4,12900,16.10,76.5,2.89,4460
3,Angola,119.0,62.3,2.85,42.9,5900,22.40,60.1,6.16,3530
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200
...,...,...,...,...,...,...,...,...,...,...
162,Vanuatu,29.2,46.6,5.25,52.7,2950,2.62,63.0,3.50,2970
163,Venezuela,17.1,28.5,4.91,17.6,16500,45.90,75.4,2.47,13500
164,Vietnam,23.3,72.0,6.84,80.2,4490,12.10,73.1,1.95,1310
165,Yemen,56.3,30.0,5.18,34.4,4480,23.60,67.5,4.67,1310


In [3]:
"""Build the model matrix: 9 numeric features, `country` held aside as the index.

`country` is an ID, never a feature. Making it the index instead of dropping it means
cluster labels can be joined back to country names later — after `drop(inplace=True)`
there was no way to say *which* countries a cluster contained.
"""
X = df.set_index('country').copy()
X.shape

(167, 9)

In [4]:
"""Log-transform income and gdpp (skew 2.23 and 2.22).

Both are strictly positive (min income 609, min gdpp 231), so a plain log is safe — no
shift, no NaN. The assert replaces the old `if x > 0 else 0` fallback: mapping a
non-positive value to 0 would put it *above* every legitimate small value once logged
(log of anything under 1 is negative), silently corrupting the distance matrix. Better
to fail loudly than to scale a poisoned column.

exports and imports are skewed too (2.45 and 1.91) but are NOT logged here — see the
next cell for why plain log is the wrong tool for them.

Writing to a new frame keeps this cell re-runnable — the old version logged `df` in
place, so a second execution gave log(log(gdpp)) with no warning.
"""
LOG_COLS = ['income', 'gdpp']

assert (X[LOG_COLS] > 0).all().all(), 'log needs strictly positive values'

X_log = X.copy()
X_log[LOG_COLS] = np.log(X_log[LOG_COLS])

In [5]:
"""Yeo-Johnson for the three skewed columns plain log can't handle.

`inflation` — skew 5.15, but 8 countries are negative (Seychelles -4.21, Ireland -3.22,
Japan -1.90, ...), so np.log returns NaN for them. Yeo-Johnson is defined on the whole
real line, which is why it's used rather than a shift-then-log.

`exports` / `imports` — skew 2.45 and 1.91. Positive, so log *runs*, but it makes things
worse: Myanmar sits at 0.109 and 0.066 against medians of 35 and 43, so log stretches the
bottom tail instead of the top one and flips the skew to -2.72 / -4.92. Scaled, Myanmar
lands at -7.5 / -9.8 — a bigger outlier than the +6.0 the raw columns produced. Measured
across raw / log / log1p / sqrt / yeo, Yeo-Johnson is the only one that gets both near
zero (0.10 / 0.17) and keeps the extremes inside +/-3.2.

Leaving these two raw was what produced the trade-entrepot cluster: logging income and
gdpp compressed them to a +/-1.4 band while untouched exports spanned +/-6.0, so Euclidean
distance ended up measuring trade openness rather than development.

`standardize=False` on purpose: RobustScaler does the centring and scaling in the next
cell, and it should see all 9 columns on the same footing.

`pt` is kept, not discarded — inverse_transform is what turns a cluster centroid back
into real percentages when the clusters need describing.
"""
from sklearn.preprocessing import PowerTransformer

YJ_COLS = ['inflation', 'exports', 'imports']

pt = PowerTransformer(method='yeo-johnson', standardize=False)

X_yj = X_log.copy()
X_yj[YJ_COLS] = pt.fit_transform(X_yj[YJ_COLS])

In [6]:
X_yj.head()

,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
country,,,,,,,,,
Afghanistan,90.2,3.403266,7.58,8.315708,7.383989,4.038811,56.2,5.82,6.315358
Albania,16.6,5.562585,6.55,8.632407,9.203316,2.505919,76.3,1.65,8.316300
Algeria,27.3,6.374882,4.17,6.998139,9.464983,5.537502,76.5,2.89,8.402904
Angola,119.0,7.775477,2.85,8.137717,8.682708,6.668443,60.1,6.16,8.169053
Antigua and Barbuda,10.3,6.843588,6.03,9.441614,9.857444,1.086089,76.8,2.13,9.409191


In [7]:
"""Scale all 9 features. K-Means is Euclidean, so on raw units income (to 125,000) would
drown total_fer (1.15-7.49) outright.

RobustScaler over StandardScaler: it centres on the median and scales by the IQR, so the
countries far out on child_mort (Haiti 208, Sierra Leone 160, Chad 150) don't inflate the
spread and squash everyone else toward zero. StandardScaler's mean and std are both
dragged by exactly those points.

`index=X_yj.index` matters — without it the DataFrame constructor silently replaces
country with a RangeIndex and the labels are gone. `scaler` is kept for inverse_transform.
"""
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X_yj),
    columns=X_yj.columns,
    index=X_yj.index,
)
X_scaled.head()

,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
country,,,,,,,,,
Afghanistan,1.316620,-1.350114,0.342391,0.055347,-0.950965,0.393415,-1.469565,1.635492,-0.904371
Albania,-0.050139,-0.280355,0.062500,0.178799,-0.001574,-0.105854,0.278261,-0.364508,-0.055359
Algeria,0.148561,0.122069,-0.584239,-0.458250,0.134973,0.881545,0.295652,0.230216,-0.018613
Angola,1.851439,0.815945,-0.942935,-0.014035,-0.273246,1.249896,-1.130435,1.798561,-0.117837
Antigua and Barbuda,-0.167131,0.354273,-0.078804,0.494232,0.339774,-0.568298,0.321739,-0.134293,0.408361


In [8]:
"""Check the transforms actually did what they were meant to, before anything is saved.

Skew should be pulled toward 0 for income, gdpp and inflation; nothing should be NaN;
the matrix should still be 167 x 9 with country as the index. Scaling is a shift and a
divide, so it doesn't change skew — the numbers below are the log/Yeo-Johnson step.
"""
check = pd.DataFrame({'raw skew': X.skew(), 'transformed skew': X_scaled.skew()}).round(2)
print(check, '\n')
print(f'shape   {X_scaled.shape}')
print(f'nulls   {int(X_scaled.isna().sum().sum())}')
print(f'index   {X_scaled.index.name}, {X_scaled.index.nunique()} unique')

            raw skew  transformed skew
child_mort      1.45              1.45
exports         2.45              0.10
health          0.71              0.71
imports         1.91              0.17
income          2.23             -0.24
inflation       5.15              0.18
life_expec     -0.97             -0.97
total_fer       0.97              0.97
gdpp            2.22              0.01 

shape   (167, 9)
nulls   0
index   country, 167 unique


In [9]:
"""Save the model matrix.

Path is relative to notebook/ and points into data/preprocessed_data/ — the old call
wrote preprocessed_data.csv into notebook/ and the file only reached data/ by hand.

`index=True` (the default) so country travels with the rows; read it back with
`pd.read_csv(OUT, index_col='country')`.
"""
OUT = Path('..') / 'data' / 'preprocessed_data' / 'preprocessed_data.csv'
OUT.parent.mkdir(parents=True, exist_ok=True)

X_scaled.to_csv(OUT)
print(f'wrote {X_scaled.shape[0]} x {X_scaled.shape[1]} to {OUT}')

wrote 167 x 9 to ..\data\preprocessed_data\preprocessed_data.csv
